# Flight Data Simulation, Processing & Analysis

This notebook implements a two-phase pipeline:

**Phase 1 — Data Generation:** create ~5000 JSON files containing randomly
generated flight records and store them under
`/tmp/flights/<MM-YY>-<origin_city>-flights.json`.

**Phase 2 — Data Analysis & Cleaning:** load all generated files, drop dirty
records, then report:
- total records processed, total dirty records, total runtime in ms
- AVG and P95 of `flight_duration_secs` for the **Top-25 destination cities**
  (ranked by total passengers arriving)
- a per-city passenger-balance analysis (max & min cities)

Python 3.7+ standard library only — no external dependencies.


## 1. Imports & Global Configuration

In [1]:
import os
import json
import random
import shutil
import time
import glob
import statistics
from datetime import datetime, timedelta
from collections import defaultdict

# Reproducibility for the random generator (comment out for fully random runs)
random.seed(42)

# --- tunable parameters --------------------------------------------------
NUM_FILES        = 5000                       # ~5000 JSON files
RECORDS_MIN      = 50                         # M ∈ [50, 100]
RECORDS_MAX      = 100
NUM_CITIES       = random.randint(100, 200)   # K ∈ [100, 200]
DIRTY_PROB       = random.uniform(0.005, 0.01)  # L ∈ [0.5%, 1%]
OUTPUT_DIR       = "/tmp/flights"
# -------------------------------------------------------------------------

print(f"NUM_FILES   : {NUM_FILES}")
print(f"NUM_CITIES  : {NUM_CITIES}")
print(f"DIRTY_PROB  : {DIRTY_PROB:.4%}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")


NUM_FILES   : 5000
NUM_CITIES  : 181
DIRTY_PROB  : 0.5557%
OUTPUT_DIR  : /tmp/flights


## 2. Build the City Pool

In [2]:
# A reasonably sized list of real city names to sample from.
CITY_POOL_FULL = [
    "New York","Los Angeles","Chicago","Houston","Phoenix","Philadelphia","San Antonio",
    "San Diego","Dallas","San Jose","Austin","Jacksonville","Fort Worth","Columbus",
    "Charlotte","San Francisco","Indianapolis","Seattle","Denver","Washington","Boston",
    "El Paso","Nashville","Detroit","Oklahoma City","Portland","Las Vegas","Memphis",
    "Louisville","Baltimore","Milwaukee","Albuquerque","Tucson","Fresno","Sacramento",
    "Kansas City","Mesa","Atlanta","Omaha","Colorado Springs","Raleigh","Miami","Oakland",
    "Minneapolis","Tulsa","Cleveland","Wichita","Arlington","London","Paris","Berlin",
    "Madrid","Rome","Amsterdam","Vienna","Prague","Warsaw","Budapest","Lisbon","Dublin",
    "Stockholm","Oslo","Copenhagen","Helsinki","Brussels","Zurich","Geneva","Athens",
    "Istanbul","Moscow","Kyiv","Tokyo","Osaka","Kyoto","Seoul","Beijing","Shanghai",
    "Guangzhou","Shenzhen","Hong Kong","Taipei","Bangkok","Singapore","Kuala Lumpur",
    "Jakarta","Manila","Hanoi","Ho Chi Minh City","Mumbai","Delhi","Bengaluru","Chennai",
    "Hyderabad","Kolkata","Pune","Ahmedabad","Karachi","Lahore","Islamabad","Dhaka",
    "Sydney","Melbourne","Brisbane","Perth","Auckland","Wellington","Toronto","Montreal",
    "Vancouver","Calgary","Ottawa","Mexico City","Guadalajara","Monterrey","Sao Paulo",
    "Rio de Janeiro","Brasilia","Buenos Aires","Lima","Bogota","Santiago","Caracas",
    "Quito","Havana","San Juan","Cairo","Lagos","Nairobi","Cape Town","Johannesburg",
    "Casablanca","Algiers","Tunis","Addis Ababa","Accra","Dakar","Riyadh","Dubai",
    "Abu Dhabi","Doha","Kuwait City","Muscat","Tehran","Baghdad","Jerusalem","Tel Aviv",
    "Beirut","Damascus","Amman","Reykjavik","Tbilisi","Yerevan","Baku","Tashkent",
    "Almaty","Ulaanbaatar","Wellington","Suva","Honolulu","Anchorage","Vancouver",
    "Calgary","Edmonton","Winnipeg","Quebec City","Halifax","San Salvador","Tegucigalpa",
    "Managua","Panama City","San Jose CR","La Paz","Asuncion","Montevideo","Sucre",
    "Belmopan","Kingston","Port-au-Prince","Santo Domingo","Nassau","Bridgetown",
    "Port of Spain","Georgetown","Paramaribo","Cayenne","Stanley","Apia","Nuku'alofa",
    "Funafuti","Tarawa","Majuro","Palikir","Yaren","Port Moresby","Honiara","Pago Pago"
]
# Ensure uniqueness and trim to NUM_CITIES
CITY_POOL_FULL = list(dict.fromkeys(CITY_POOL_FULL))   # keep order, dedupe
CITY_POOL      = random.sample(CITY_POOL_FULL, NUM_CITIES)
print(f"Sampled {len(CITY_POOL)} cities. First 10: {CITY_POOL[:10]}")


Sampled 181 cities. First 10: ['Yaren', 'Kyiv', 'Copenhagen', 'Budapest', 'Kansas City', 'Las Vegas', 'Kingston', 'Doha', 'Nashville', 'Yerevan']


## 3. Phase 1 — Data Generation

Each file:
* contains a JSON array with M ∈ [50, 100] flight records
* is named `<MM-YY>-<origin_city>-flights.json`
* lives in `/tmp/flights/`

About 0.5%–1% of records have one or more fields randomly set to `null`
(those are the *dirty* records Phase 2 must filter out).


In [3]:
def random_iso_date(year, month):
    """A random YYYY-MM-DD inside the given month/year."""
    if month == 12:
        next_month = datetime(year + 1, 1, 1)
    else:
        next_month = datetime(year, month + 1, 1)
    first = datetime(year, month, 1)
    delta_days = (next_month - first).days
    return (first + timedelta(days=random.randint(0, delta_days - 1))).strftime("%Y-%m-%d")


FIELDS = ["date", "origin_city", "destination_city",
          "flight_duration_secs", "#_of_passengers_on_board"]


def make_record(origin_city, year, month):
    """Build one flight record, possibly dirtying it."""
    destination_city = random.choice([c for c in CITY_POOL if c != origin_city])
    rec = {
        "date": random_iso_date(year, month),
        "origin_city": origin_city,
        "destination_city": destination_city,
        "flight_duration_secs": random.randint(30 * 60, 16 * 60 * 60),  # 30 min – 16 hr
        "#_of_passengers_on_board": random.randint(20, 350),
    }
    # With probability DIRTY_PROB, null out one or more fields
    if random.random() < DIRTY_PROB:
        n_null = random.randint(1, 2)
        for fld in random.sample(FIELDS, n_null):
            rec[fld] = None
    return rec


# Fresh output directory
if os.path.isdir(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

now      = datetime.now()
mmyy_tag = now.strftime("%m-%y")
year, month = now.year, now.month

gen_start = time.perf_counter()
total_records = 0
total_dirty   = 0

# We want ~NUM_FILES files. We loop, picking an origin_city; one file per
# (origin_city, sequence) so the same city can show up multiple times.
# To avoid filename collisions we append a numeric suffix.
city_file_counter = defaultdict(int)

for i in range(NUM_FILES):
    origin_city = random.choice(CITY_POOL)
    city_file_counter[origin_city] += 1
    seq = city_file_counter[origin_city]
    # Sanitise the city for use in a filename
    safe_city = origin_city.replace(" ", "_").replace("'", "")
    fname     = f"{mmyy_tag}-{safe_city}-flights_{seq}.json"
    fpath     = os.path.join(OUTPUT_DIR, fname)

    n_records = random.randint(RECORDS_MIN, RECORDS_MAX)
    records   = [make_record(origin_city, year, month) for _ in range(n_records)]
    total_records += len(records)
    total_dirty   += sum(1 for r in records if any(v is None for v in r.values()))

    with open(fpath, "w") as fh:
        json.dump(records, fh)

gen_elapsed = time.perf_counter() - gen_start

print(f"Files written : {NUM_FILES}")
print(f"Records total : {total_records}")
print(f"Dirty records : {total_dirty}  ({total_dirty/total_records:.3%})")
print(f"Generation runtime : {gen_elapsed*1000:.1f} ms")

# Sanity check on the filesystem
disk_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.json")))
print(f"Files on disk : {len(disk_files)}")
print(f"Sample file   : {disk_files[0]}")


Files written : 5000
Records total : 374808
Dirty records : 2162  (0.577%)
Generation runtime : 4077.2 ms
Files on disk : 5000
Sample file   : /tmp/flights/05-26-Abu_Dhabi-flights_1.json


### Inspect a sample generated file

In [4]:
sample_path = disk_files[0]
with open(sample_path) as fh:
    sample = json.load(fh)
print(f"{sample_path}\n  -> {len(sample)} records, first 2:")
print(json.dumps(sample[:2], indent=2))


/tmp/flights/05-26-Abu_Dhabi-flights_1.json
  -> 56 records, first 2:
[
  {
    "date": "2026-05-06",
    "origin_city": "Abu Dhabi",
    "destination_city": "Damascus",
    "flight_duration_secs": 5549,
    "#_of_passengers_on_board": 170
  },
  {
    "date": "2026-05-12",
    "origin_city": "Abu Dhabi",
    "destination_city": "Indianapolis",
    "flight_duration_secs": 9272,
    "#_of_passengers_on_board": 167
  }
]


## 4. Phase 2 — Data Analysis & Cleaning

Steps:
1. Scan every `*.json` file under `/tmp/flights/`.
2. Reject any record that has a NULL in *any* required field (dirty record).
3. Aggregate:
   * **Top-25 destination cities** ranked by total passengers arriving;
     for those compute AVG & P95 of `flight_duration_secs`.
   * **Passenger balance** per city: `arrivals - departures`.


In [5]:
REQUIRED_FIELDS = ("date", "origin_city", "destination_city",
                   "flight_duration_secs", "#_of_passengers_on_board")


def is_dirty(rec):
    """A record is dirty if it is missing a field or any value is None."""
    return any(rec.get(f) is None for f in REQUIRED_FIELDS)


def percentile(values, pct):
    """P-th percentile via nearest-rank (no numpy dependency)."""
    if not values:
        return 0.0
    values = sorted(values)
    k = max(0, min(len(values) - 1, int(round((pct / 100.0) * len(values) + 0.5)) - 1))
    return values[k]


analysis_start = time.perf_counter()

records_processed = 0
dirty_count       = 0

# Per-destination passenger totals & durations for AVG/P95
dest_passengers   = defaultdict(int)
dest_durations    = defaultdict(list)

# Per-city balance: arrivals minus departures
balance           = defaultdict(int)

for fpath in glob.iglob(os.path.join(OUTPUT_DIR, "*.json")):
    with open(fpath) as fh:
        recs = json.load(fh)
    for rec in recs:
        records_processed += 1
        if is_dirty(rec):
            dirty_count += 1
            continue   # skip the dirty record
        oc = rec["origin_city"]
        dc = rec["destination_city"]
        dur = rec["flight_duration_secs"]
        pax = rec["#_of_passengers_on_board"]

        dest_passengers[dc] += pax
        dest_durations[dc].append(dur)

        balance[oc] -= pax     # passengers leave origin
        balance[dc] += pax     # passengers arrive at destination

analysis_elapsed_ms = (time.perf_counter() - analysis_start) * 1000

print(f"Records processed      : {records_processed}")
print(f"Dirty records          : {dirty_count}  ({dirty_count/records_processed:.3%})")
print(f"Analysis runtime (ms)  : {analysis_elapsed_ms:.1f}")


Records processed      : 374808
Dirty records          : 2162  (0.577%)
Analysis runtime (ms)  : 739.9


### 4.1 AVG & P95 flight duration for Top-25 destination cities

In [6]:
top25 = sorted(dest_passengers.items(), key=lambda kv: kv[1], reverse=True)[:25]

print(f"{'Rank':<5}{'Destination City':<22}{'Passengers':>12}{'AVG (s)':>12}{'P95 (s)':>12}")
print("-" * 63)
for rank, (city, pax_total) in enumerate(top25, start=1):
    durs = dest_durations[city]
    avg  = statistics.mean(durs)
    p95  = percentile(durs, 95)
    print(f"{rank:<5}{city:<22}{pax_total:>12,d}{avg:>12,.1f}{p95:>12,d}")


Rank Destination City        Passengers     AVG (s)     P95 (s)
---------------------------------------------------------------
1    Atlanta                    405,777    29,665.8      54,283
2    Miami                      402,281    29,408.4      55,157
3    Auckland                   400,815    29,790.7      55,158
4    Cape Town                  399,383    30,005.1      55,349
5    Muscat                     399,025    29,561.6      54,632
6    Winnipeg                   396,476    29,505.7      54,735
7    Havana                     395,777    29,733.3      54,994
8    Hanoi                      395,587    29,174.3      54,926
9    Pune                       395,586    30,000.3      54,856
10   Abu Dhabi                  395,174    30,107.7      54,616
11   Majuro                     395,051    30,108.5      54,806
12   Halifax                    395,041    30,366.5      54,544
13   Georgetown                 394,828    29,543.0      54,580
14   Honiara                    394,783 

### 4.2 Passenger Balance Analysis (max / min remaining)

In [7]:
# Every city in the pool starts at 0 — make sure cities with no traffic appear too
for c in CITY_POOL:
    balance.setdefault(c, 0)

max_city, max_val = max(balance.items(), key=lambda kv: kv[1])
min_city, min_val = min(balance.items(), key=lambda kv: kv[1])

print(f"City with MAX remaining passengers : {max_city!r:25}  net = {max_val:+,d}")
print(f"City with MIN remaining passengers : {min_city!r:25}  net = {min_val:+,d}")

print("\nTop 10 net-positive cities:")
for c, v in sorted(balance.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(f"  {c:<22} {v:+,d}")

print("\nBottom 10 net-negative cities:")
for c, v in sorted(balance.items(), key=lambda kv: kv[1])[:10]:
    print(f"  {c:<22} {v:+,d}")


City with MAX remaining passengers : 'Paris'                    net = +277,056
City with MIN remaining passengers : 'Oklahoma City'            net = -197,232

Top 10 net-positive cities:
  Paris                  +277,056
  Georgetown             +242,500
  Santo Domingo          +203,769
  Jacksonville           +168,086
  Tel Aviv               +144,297
  Honiara                +131,789
  Minneapolis            +115,977
  Wellington             +107,619
  Auckland               +104,813
  Honolulu               +103,132

Bottom 10 net-negative cities:
  Oklahoma City          -197,232
  Tunis                  -172,397
  Hyderabad              -159,600
  Quebec City            -150,424
  Palikir                -135,516
  Omaha                  -133,113
  Tbilisi                -131,871
  Jerusalem              -127,324
  Jakarta                -125,904
  Kansas City            -113,431


## 5. Final Summary

In [8]:
summary = {
    "phase_1": {
        "files_generated"       : NUM_FILES,
        "records_generated"     : total_records,
        "dirty_records_generated": total_dirty,
        "city_pool_size"        : NUM_CITIES,
        "dirty_probability"     : round(DIRTY_PROB, 5),
        "generation_runtime_ms" : round(gen_elapsed * 1000, 1),
        "output_directory"      : OUTPUT_DIR,
    },
    "phase_2": {
        "records_processed"     : records_processed,
        "dirty_records"         : dirty_count,
        "analysis_runtime_ms"   : round(analysis_elapsed_ms, 1),
        "max_balance_city"      : {"city": max_city, "net_passengers": max_val},
        "min_balance_city"      : {"city": min_city, "net_passengers": min_val},
    },
}
print(json.dumps(summary, indent=2))


{
  "phase_1": {
    "files_generated": 5000,
    "records_generated": 374808,
    "dirty_records_generated": 2162,
    "city_pool_size": 181,
    "dirty_probability": 0.00556,
    "generation_runtime_ms": 4077.2,
    "output_directory": "/tmp/flights"
  },
  "phase_2": {
    "records_processed": 374808,
    "dirty_records": 2162,
    "analysis_runtime_ms": 739.9,
    "max_balance_city": {
      "city": "Paris",
      "net_passengers": 277056
    },
    "min_balance_city": {
      "city": "Oklahoma City",
      "net_passengers": -197232
    }
  }
}
